## Assignment 1: Multilingual Embedding-based Machine Translation

## 1 Task) 13 points

**In this homework** **<font color='red'>YOU</font>** will make machine translation system without using parallel corpora, alignment, attention, 100500 depth super-cool recurrent neural network and all that kind superstuff.

But even without parallel corpora this system can be good enough (hopefully).

For our system we choose two kindred Slavic languages: Ukrainian and Russian.

# PLEASE CHECK ASSERT, BE SURE THAT YOU IMPLEMENTED ALL THE CODE CORRECT IN FIRST PART

### Feel the difference!

(_синій кіт_ vs. _синій кит_)

![blue_cat_blue_whale.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/blue_cat_blue_whale.png)

### Fragment of the Swadesh list for some slavic languages

The Swadesh list is a lexicostatistical stuff. It's named after American linguist Morris Swadesh and contains basic lexis. This list are used to define subgroupings of languages, its relatedness.

So we can see some kind of word invariance for different Slavic languages.


| Russian         | Belorussian              | Ukrainian               | Polish             | Czech                         | Bulgarian            |
|-----------------|--------------------------|-------------------------|--------------------|-------------------------------|-----------------------|
| женщина         | жанчына, кабета, баба    | жінка                   | kobieta            | žena                          | жена                  |
| мужчина         | мужчына                  | чоловік, мужчина        | mężczyzna          | muž                           | мъж                   |
| человек         | чалавек                  | людина, чоловік         | człowiek           | člověk                        | човек                 |
| ребёнок, дитя   | дзіця, дзіцёнак, немаўля | дитина, дитя            | dziecko            | dítě                          | дете                  |
| жена            | жонка                    | дружина, жінка          | żona               | žena, manželka, choť          | съпруга, жена         |
| муж             | муж, гаспадар            | чоловiк, муж            | mąż                | muž, manžel, choť             | съпруг, мъж           |
| мать, мама      | маці, матка              | мати, матір, неня, мама | matka              | matka, máma, 'стар.' mateř    | майка                 |
| отец, тятя      | бацька, тата             | батько, тато, татусь    | ojciec             | otec                          | баща, татко           |
| много           | шмат, багата             | багато                  | wiele              | mnoho, hodně                  | много                 |
| несколько       | некалькі, колькі         | декілька, кілька        | kilka              | několik, pár, trocha          | няколко               |
| другой, иной    | іншы                     | інший                   | inny               | druhý, jiný                   | друг                  |
| зверь, животное | жывёла, звер, істота     | тварина, звір           | zwierzę            | zvíře                         | животно               |
| рыба            | рыба                     | риба                    | ryba               | ryba                          | риба                  |
| птица           | птушка                   | птах, птиця             | ptak               | pták                          | птица                 |
| собака, пёс     | сабака                   | собака, пес             | pies               | pes                           | куче, пес             |
| вошь            | вош                      | воша                    | wesz               | veš                           | въшка                 |
| змея, гад       | змяя                     | змія, гад               | wąż                | had                           | змия                  |
| червь, червяк   | чарвяк                   | хробак, черв'як         | robak              | červ                          | червей                |
| дерево          | дрэва                    | дерево                  | drzewo             | strom, dřevo                  | дърво                 |
| лес             | лес                      | ліс                     | las                | les                           | гора, лес             |
| палка           | кій, палка               | палиця                  | patyk, pręt, pałka | hůl, klacek, prut, kůl, pálka | палка, пръчка, бастун |

But the context distribution of these languages demonstrates even more invariance. And we can use this fact for our for our purposes.

## Data

In [ ]:
!pip install fasttext

In [22]:
import gensim
import numpy as np
from gensim.models import KeyedVectors


Download embeddings here:
* [cc.uk.300.vec.zip](https://www.dropbox.com/scl/fo/czrdhfndzvirs0c3x4adh/AIDkoaHAyd3bcpAegJ4bz3A?rlkey=f7xnl9lqiahjbor3ucxyy6u6p&st=ee70k4ik&dl=0)
* [cc.ru.300.vec.zip](https://www.dropbox.com/scl/fo/czrdhfndzvirs0c3x4adh/AIDkoaHAyd3bcpAegJ4bz3A?rlkey=f7xnl9lqiahjbor3ucxyy6u6p&st=ee70k4ik&dl=0)

Load embeddings for ukrainian and russian.

In [ ]:
uk_emb = KeyedVectors.load_word2vec_format("cc.uk.300.vec", encoding='utf-8', limit=None)

In [ ]:
ru_emb = KeyedVectors.load_word2vec_format("cc.ru.300.vec", encoding='utf-8', limit=None)

In [ ]:
ru_emb.most_similar([ru_emb["август"]], topn=10)

[('август', 1.0000001192092896),
 ('июль', 0.9383152723312378),
 ('сентябрь', 0.9240029454231262),
 ('июнь', 0.9222574830055237),
 ('октябрь', 0.9095539450645447),
 ('ноябрь', 0.8930036425590515),
 ('апрель', 0.8729087114334106),
 ('декабрь', 0.8652557730674744),
 ('март', 0.8545795679092407),
 ('февраль', 0.8401415944099426)]

In [ ]:
uk_emb.most_similar([uk_emb["серпень"]])

[('серпень', 0.9999998807907104),
 ('липень', 0.9096441268920898),
 ('вересень', 0.9016969203948975),
 ('червень', 0.8992518782615662),
 ('жовтень', 0.8810408115386963),
 ('листопад', 0.8787633180618286),
 ('квітень', 0.8592804670333862),
 ('грудень', 0.8586863279342651),
 ('травень', 0.840811014175415),
 ('лютий', 0.8256431221961975)]

Load small dictionaries for correspoinding words pairs as trainset and testset.

In [ ]:
def load_word_pairs(filename):
    uk_ru_pairs = []
    uk_vectors = []
    ru_vectors = []
    with open(filename, "r") as inpf:
        for line in inpf:
            uk, ru = line.rstrip().split("\t")
            if uk not in uk_emb or ru not in ru_emb:
                continue
            uk_ru_pairs.append((uk, ru))
            uk_vectors.append(uk_emb[uk])
            ru_vectors.append(ru_emb[ru])
    return uk_ru_pairs, np.array(uk_vectors), np.array(ru_vectors)

In [18]:
uk_ru_train, X_train, Y_train = load_word_pairs("ukr_rus.train.txt")
print(uk_ru_train, X_train, Y_train)

NameError: name 'load_word_pairs' is not defined

In [ ]:
uk_ru_test, X_test, Y_test = load_word_pairs("ukr_rus.test.txt")

## Embedding space mapping

Let $x_i \in \mathrm{R}^d$ be the distributed representation of word $i$ in the source language, and $y_i \in \mathrm{R}^d$ is the vector representation of its translation. Our purpose is to learn such linear transform $W$ that minimizes euclidian distance between $Wx_i$ and $y_i$ for some subset of word embeddings. Thus we can formulate so-called Procrustes problem:

$$W^*= \arg\min_W \sum_{i=1}^n||Wx_i - y_i||_2$$
or
$$W^*= \arg\min_W ||WX - Y||_F$$

where $||*||_F$ - Frobenius norm.

In Greek mythology, Procrustes or "the stretcher" was a rogue smith and bandit from Attica who attacked people by stretching them or cutting off their legs, so as to force them to fit the size of an iron bed. We make same bad things with source embedding space. Our Procrustean bed is target embedding space.

![embedding_mapping.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/embedding_mapping.png)

![procrustes.png](https://github.com/yandexdataschool/nlp_course/raw/master/resources/procrustes.png)

But wait...$W^*= \arg\min_W \sum_{i=1}^n||Wx_i - y_i||_2$ looks like simple multiple linear regression (without intercept fit). So let's code.

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

mapping = LinearRegression().fit(X_train, Y_train)
# Y_test = mapping.predict(X_test)

Let's take a look at neigbours of the vector of word _"серпень"_ (_"август"_ in Russian) after linear transform.

In [ ]:
august = regressor.predict(uk_emb["серпень"].reshape(1, -1))
ru_emb.most_similar(august)

[('апрель', 0.8541286587715149),
 ('июнь', 0.8411202430725098),
 ('март', 0.839699387550354),
 ('сентябрь', 0.8359869718551636),
 ('февраль', 0.832929790019989),
 ('октябрь', 0.8311845660209656),
 ('ноябрь', 0.8278924226760864),
 ('июль', 0.823452889919281),
 ('август', 0.8120501637458801),
 ('декабрь', 0.803900420665741)]

We can see that neighbourhood of this embedding cosists of different months, but right variant is on the ninth place.

As quality measure we will use precision top-1, top-5 and top-10 (for each transformed Ukrainian embedding we count how many right target pairs are found in top N nearest neighbours in Russian embedding space).

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def precision(pairs, mapped_vectors, topn=1):
    """
    Calculate precision for top-N nearest neighbors.

    :args:
        pairs (list of tuples): List of right word pairs [(uk_word_0, ru_word_0), ...]
        mapped_vectors (list of numpy arrays): List of embeddings after mapping from source embedding space to destination
        topn (int): Number of nearest neighbors in destination embedding space to check against

    :returns:
        precision_val (float): Precision at top N, the proportion of words for which the correct translation
                               appears in the top N nearest neighbors.
    """
    assert len(pairs) == len(mapped_vectors)
    num_matches = 0
    for i, (_, ru) in enumerate(pairs):
        for (f, s) in ru_emb.most_similar(mapped_vectors[i].reshape(1, -1), topn=topn):
            if ru == f:
                num_matches += 1
    precision_val = num_matches / len(pairs)
    print(precision_val)
    return precision_val


In [ ]:
assert precision([("серпень", "август")], august, topn=5) == 0.0
assert precision([("серпень", "август")], august, topn=9) == 1.0
assert precision([("серпень", "август")], august, topn=10) == 1.0

0.0
1.0
1.0


In [ ]:
assert precision(uk_ru_test, X_test) == 0.0
assert precision(uk_ru_test, Y_test) == 1.0

0.0
1.0


In [ ]:
precision_top1 = precision(uk_ru_test, mapping.predict(X_test), 1)
precision_top5 = precision(uk_ru_test, mapping.predict(X_test), 5)

assert precision_top1 >= 0.635
assert precision_top5 >= 0.811

0.6356589147286822
0.8113695090439277


In [ ]:
def evaluate_mapping(X_test, Y_test, src_emb, tgt_emb, W, topn=10):
    correct_top1 = 0
    correct_top5 = 0
    correct_top10 = 0
    total = len(X_test)

    for i in range(total):
        # Преобразование украинского вектора
        transformed_vector = W @ X_test[i]

        # Try to get the target word vector, handle KeyError if not found
        try:
            target_word = [word for word, _ in uk_ru_test][i] # Get the target word from uk_ru_test
            target_word_vector = tgt_emb[target_word]
        except KeyError:
            #print(f"Warning: Word '{target_word}' not found in target embeddings. Skipping.")
            continue  # Skip this word if not found in embeddings

        # Поиск ближайших слов в русском пространстве
        similar_words = [w for w, _ in tgt_emb.most_similar([transformed_vector], topn=topn)]

        # Сравнение с правильным переводом
        if target_word in similar_words[:1]: # Compare with the actual target word
            correct_top1 += 1
        if target_word in similar_words[:5]:
            correct_top5 += 1
        if target_word in similar_words[:10]:
            correct_top10 += 1

    return {
        "top1": correct_top1 / total,
        "top5": correct_top5 / total,
        "top10": correct_top10 / total
    }

## Making it better (orthogonal Procrustean problem)

It can be shown (see original paper) that a self-consistent linear mapping between semantic spaces should be orthogonal.
We can restrict transform $W$ to be orthogonal. Then we will solve next problem:

$$W^*= \arg\min_W ||WX - Y||_F \text{, where: } W^TW = I$$

$$I \text{- identity matrix}$$

Instead of making yet another regression problem we can find optimal orthogonal transformation using singular value decomposition. It turns out that optimal transformation $W^*$ can be expressed via SVD components:
$$X^TY=U\Sigma V^T\text{, singular value decompostion}$$
$$W^*=UV^T$$

In [ ]:
import numpy as np
from numpy.linalg import svd

def learn_transform(X_train, Y_train):
    """
    :returns: W* : float matrix[emb_dim x emb_dim] as defined in the formula above
    """
    u, s, vh = svd(np.matmul(X_train.T, Y_train), full_matrices=True)
    return np.matmul(u, vh)

In [ ]:
W = learn_transform(X_train, Y_train)

In [ ]:
ru_emb.most_similar([np.matmul(uk_emb["серпень"], W)])

[('апрель', 0.8237906694412231),
 ('сентябрь', 0.8049713373184204),
 ('март', 0.8025653958320618),
 ('июнь', 0.8021842241287231),
 ('октябрь', 0.8001735806465149),
 ('ноябрь', 0.7934483289718628),
 ('февраль', 0.7914120554924011),
 ('июль', 0.7908109426498413),
 ('август', 0.7891016602516174),
 ('декабрь', 0.7686373591423035)]

In [ ]:
assert precision(uk_ru_test, np.matmul(X_test, W)) >= 0.653
assert precision(uk_ru_test, np.matmul(X_test, W), 5) >= 0.824

0.6537467700258398
0.8242894056847545


## UK-RU Translator

Now we are ready to make simple word-based translator: for each word in source language in shared embedding space we find the nearest in target language.


In [ ]:
with open("fairy_tale.txt", "r") as inpf:
    uk_sentences = [line.rstrip().lower() for line in inpf]

In [ ]:
import numpy as np
import nltk
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import WordPunctTokenizer

def translate(sentence):
    """
    :args:
        sentence - sentence in Ukrainian (str)
    :returns:
        translation - sentence in Russian (str)

    * find ukrainian embedding for each word in sentence
    * transform ukrainian embedding vector
    * find nearest russian word and replace
    """
    # YOUR CODE HERE
    tokenizer = WordPunctTokenizer()
    data_tok = [j.lower() for j in tokenizer.tokenize(sentence)]
    data_trans = ([ru_emb.most_similar([np.matmul(uk_emb[i], W)], topn=1)[0][0] if i in uk_emb.key_to_index else '' for i in data_tok])
    print(data_trans)
    return ' '.join(data_trans)

In [ ]:
assert translate(".") == "."
assert translate("1 , 3") == "1 , 3"
assert translate("кіт зловив мишу") == "кот поймал мышку"

['.']
['1', ',', '3']
['кот', 'поймал', 'мышку']


In [ ]:
for sentence in uk_sentences:
    print("src: {}\ndst: {}\n".format(sentence, translate(sentence)))

['лисичка', '–', 'сестричка', 'и', 'волк', '–', '']
src: лисичка - сестричка і вовк - панібрат
dst: лисичка – сестричка и волк – 

['как', 'была', 'себе', 'лисичка', 'и', 'сделала', 'избушку', ',', 'и', 'и', 'живет', '.', 'а', 'оно', 'приходят', 'морозы', '.', 'из', 'лисичка', 'замерзла', 'и', 'и', 'побежала', 'во', 'село', 'огня', '', ',', 'чтобы', '', '.', 'прибегает', 'к', 'одной', 'бабы', 'и', 'и', 'говорит', ':']
src: як була собі лисичка та зробила хатку, та й живе. а це приходять холоди. от лисичка замерзла та й побігла в село вогню добувать, щоб витопити. прибігає до одної баби та й каже:
dst: как была себе лисичка и сделала избушку , и и живет . а оно приходят морозы . из лисичка замерзла и и побежала во село огня  , чтобы  . прибегает к одной бабы и и говорит :

['—', 'здоровые', 'были', ',', 'бабушку', '!', 'со', 'воскресеньем', '...', '', 'мне', 'огня', ',', 'мной', 'тебе', '', '.']
src: — здорові були, бабусю! з неділею... позичте мені огню, я вам одслужу.
dst: — здоровые 

Not so bad, right? We can easily improve translation using language model and not one but several nearest neighbours in shared embedding space. But next time.

## 2 Task)  7 points

###  Download embedding from here: https://fasttext.cc/docs/en/crawl-vectors.html (download text format). Depending on the group, choose the correct Slavic language embedding. After downloading and using the embedding, repeat all the tasks from first part, but you dont need set asserts in this task and generate the same simple text (at least 200 words) and translate it into Russian.


In [ ]:
cs_emb = KeyedVectors.load_word2vec_format("cc.cs.300.vec")

FileNotFoundError: [Errno 2] No such file or directory: 'cc.cs.300.vec'

src: лисичка - сестричка і вовк - панібрат
dst: согрет пополнены Теплообмен ратифицировали Авторизация пополнены панібрат

src: як була собі лисичка та зробила хатку, та й живе. а це приходять холоди. от лисичка замерзла та й побігла в село вогню добувать, щоб витопити. прибігає до одної баби та й каже:
dst: аммо забратьКак DIMMШкаф согрет НАТРИЯ уходуСифоны хатку, НАТРИЯ Нобоа живе. Нобоа побитый Редкол холоди. территориальными согрет ВФСК НАТРИЯ Нобоа гладильные играх.Официальная завершающих дах добувать, грнкупитьДлина витопити. U12 мл1080.- GNU 2017КонтактыНормативная НАТРИЯ Нобоа каже:

src: — здорові були, бабусю! з неділею... позичте мені огню, я вам одслужу.
dst: законсервированы Enciclopedia були, бабусю! р-он неділею... позичте МТРНовости огню, перевозимый компанииКонтакты8 одслужу.

src: — добре, — каже, — лисичко - сестричко. сідай погрійся трохи, поки я пиріжечки повибираю з печі!
dst: законсервированы добре, законсервированы каже, законсервированы лисичко пополнены сестри

## Would you like to learn more?

### Articles:
* [Exploiting Similarities among Languages for Machine Translation](https://arxiv.org/pdf/1309.4168)  - entry point for multilingual embedding studies by Tomas Mikolov (the author of W2V)
* [Offline bilingual word vectors, orthogonal transformations and the inverted softmax](https://arxiv.org/pdf/1702.03859) - orthogonal transform for unsupervised MT
* [Word Translation Without Parallel Data](https://arxiv.org/pdf/1710.04087)
* [Loss in Translation: Learning Bilingual Word Mapping with a Retrieval Criterion](https://arxiv.org/pdf/1804.07745)
* [Unsupervised Alignment of Embeddings with Wasserstein Procrustes](https://arxiv.org/pdf/1805.11222)

### Repos (with ready-to-use multilingual embeddings):
* https://github.com/facebookresearch/MUSE

* https://github.com/Babylonpartners/fastText_multilingual -